[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassermakram/tiny-llm-education/blob/main/stage1_b_word_reference.ipynb)

# Stage 1b: Zero-Layer Transformer (Word-Level) 🗺️
## target: Dialect Subway Lines (Bigram Transition Graphs)

Now that we understand how character transition metrics work, let's step up to the **Word Level**.

In a word-level language model, the vocabulary is composed of whole words.
Because we are predicting word-to-word transitions, training a full PyTorch model can take much longer due to the large vocabulary size. 

Since we already proved in Stage 1a that training a Zero-Layer network converges to the statistical bigram counts, we will **skip the training loop** and directly calculate the optimal matrices using direct counts!

### 💡 What we are showing:
We will map out the "subway lines" of the dialects. By filtering for the strongest transitions of core grammatical words (like 'الذي' in MSA vs 'اللي' in Masri), we can visualize the structural paths of each dialect as a directed graph.


In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q plotly pandas datasets networkx


In [ ]:
# 📦 Load datasets & split into words
from datasets import load_dataset
import re

print("Loading Wikipedia streams...")
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
masri_stream = load_dataset("wikimedia/wikipedia", "20231101.arz", split="train", streaming=True)

def collect_words(stream, max_words=50000):
    words = []
    for article in stream:
        # Clean text
        text = article['text'].lower()
        text = re.sub(r'[^\s\u0621-\u064A]', ' ', text)
        words.extend(text.split())
        if len(words) >= max_words:
            break
    return words[:max_words]

print("Collecting MSA words...")
msa_words = collect_words(msa_stream)
print("Collecting Masri words...")
masri_words = collect_words(masri_stream)
print(f"Collected {len(msa_words)} MSA words and {len(masri_words)} Masri words.")
print("Masri words sample:", masri_words[:30])


In [ ]:
# 🧮 Compute Bigram Counts & Transition Probabilities Directly
from collections import Counter
import pandas as pd

def compute_bigram_transitions(words, top_n=1000):
    # Find top vocabulary words
    word_counts = Counter(words)
    vocab = [w for w, c in word_counts.most_common(top_n)]
    vocab_set = set(vocab)
    
    # Calculate transitions
    transitions = Counter()
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        if w1 in vocab_set and w2 in vocab_set:
            transitions[(w1, w2)] += 1
            
    # Build transition probability dictionary
    # For each w1, what is the probability of w2?
    w1_totals = Counter()
    for (w1, w2), count in transitions.items():
        w1_totals[w1] += count
        
    probs = {}
    for (w1, w2), count in transitions.items():
        probs[(w1, w2)] = count / w1_totals[w1]
        
    return probs, vocab

print("Computing MSA bigrams...")
msa_probs, msa_vocab = compute_bigram_transitions(msa_words)
print("Computing Masri bigrams...")
masri_probs, masri_vocab = compute_bigram_transitions(masri_words)


## 🗺️ Building the Dialect Subway Map
We'll select key "seed" words that capture grammar or common starting points (like 'في', 'من', 'اللي', 'الذي', 'هو', 'هي') and trace the most probable next-word paths to draw a subway map of the language!


In [ ]:
# 🕸️ Plot the Transition Network Graph using NetworkX & Plotly
import networkx as nx
import plotly.graph_objects as go

def build_transition_graph(probs, seeds, max_depth=3, threshold=0.1):
    G = nx.DiGraph()
    visited = set()
    queue = [(seed, 0) for seed in seeds]
    
    while queue:
        node, depth = queue.pop(0)
        if depth >= max_depth or node in visited:
            continue
        visited.add(node)
        
        # Find transitions from this word
        node_transitions = {w2: p for (w1, w2), p in probs.items() if w1 == node and p >= threshold}
        
        # Sort and take top 3
        sorted_trans = sorted(node_transitions.items(), key=lambda x: x[1], reverse=True)[:3]
        for next_node, p in sorted_trans:
            G.add_edge(node, next_node, weight=p)
            queue.append((next_node, depth + 1))
            
    return G

def plot_graph_plotly(G, title):
    # Set spring layout coordinates
    pos = nx.spring_layout(G, seed=42)
    
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1.5, color='#888'),
        hoverinfo='none',
        mode='lines')

    node_x = []
    node_y = []
    node_text = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        hoverinfo='text',
        text=node_text,
        textposition="top center",
        marker=dict(
            showscale=False,
            color='skyblue',
            size=24,
            line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace],
                 layout=go.Layout(
                    title=title,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                 )
    fig.show()

# MSA seed words
msa_seeds = ['الذي', 'في', 'كان']
msa_G = build_transition_graph(msa_probs, msa_seeds)
plot_graph_plotly(msa_G, "MSA Word-Level Transition Subway Lines")

# Masri seed words
masri_seeds = ['اللي', 'في', 'كان']
masri_G = build_transition_graph(masri_probs, masri_seeds)
plot_graph_plotly(masri_G, "Masri Word-Level Transition Subway Lines")
